# Academy Award Statistical Prediction Model - Backtest

This notebook duplicates the same statistics and scoring functions from `oscars_prediction_system.ipynb`. It checks how accurate those functions are by testing them against every Oscar ceremony in the historical data.

**Method: leave-one-year-out (LOYO).** For each year being tested, the win-rates are calculated using data from every year except the year being tested. This matters because it means a year's own outcome never leaks into the data being used to calculate scores. If it somehow leaked, the accuracy could skew and be less accurate.

In [1]:
import pandas

In [2]:
# Create Data Frames for all Categories
best_picture_df = pandas.read_csv("../awards_historical_data/best_picture_history.csv")
best_actor_df = pandas.read_csv("../awards_historical_data/best_actor_history.csv")
best_actress_df = pandas.read_csv("../awards_historical_data/best_actress_history.csv")
best_supporting_actor_df = pandas.read_csv("../awards_historical_data/best_supporting_actor_history.csv")
best_supporting_actress_df = pandas.read_csv("../awards_historical_data/best_supporting_actress_history.csv")

## Statistics and Scoring

Same functions as `oscars_prediction_system.ipynb` - duplicated here on purpose rather than imported, so this notebook stays fully self-contained.

In [3]:
# Assemble all stats for Best Picture
def get_best_picture_statistics(best_picture_df):

  # Gather PGA Statistics
  pga_winners = best_picture_df[best_picture_df['won_pga'] == 1]
  pga_to_oscar_rate = pga_winners['won_oscar'].mean()

  # Gather DGA Statistics
  dga_winners = best_picture_df[best_picture_df['won_dga'] == 1]
  dga_to_oscar_rate = dga_winners['won_oscar'].mean()

  # Gather BAFTA Statistics
  bafta_winners = best_picture_df[best_picture_df['won_bafta'] == 1]
  bafta_to_oscar_rate = bafta_winners['won_oscar'].mean()

  # Gather Critic's Choice Statistics
  critics_winners = best_picture_df[best_picture_df['won_critics'] == 1]
  critics_to_oscar_rate = critics_winners['won_oscar'].mean()

  # Gather Average Nominations for Winners
  oscar_winners = best_picture_df[best_picture_df['won_oscar'] == 1]
  oscar_winner_average_nomination_count = oscar_winners['total_oscar_nominations'].mean()

  # Gather Average Nominations for Non-Winners
  non_oscar_winners = best_picture_df[best_picture_df['won_oscar'] == 0]
  non_oscar_winner_average_nomination_count = non_oscar_winners['total_oscar_nominations'].mean()

  return {"pga_rate" : pga_to_oscar_rate,
          "dga_rate" : dga_to_oscar_rate,
          "bafta_rate" : bafta_to_oscar_rate,
          "critics_rate" : critics_to_oscar_rate,
          "average_winner_nominations" : oscar_winner_average_nomination_count,
          "average_non_winner_nominations" : non_oscar_winner_average_nomination_count}

In [4]:
# Assemble all stats for acting categories
def get_acting_category_statistics(category_df):

  # Gather SAG Statistics
  sag_winners = category_df[category_df['won_sag'] == 1]
  sag_to_oscar_rate = sag_winners['won_oscar'].mean()

  # Gather Globe Statistics
  globe_winners = category_df[category_df['won_globe'] == 1]
  globe_to_oscar_rate = globe_winners['won_oscar'].mean()

  # Gather BAFTA Statistics
  bafta_winners = category_df[category_df['won_bafta'] == 1]
  bafta_to_oscar_rate = bafta_winners['won_oscar'].mean()

  # Gather Critic's Choice Statistics
  critics_winners = category_df[category_df['won_critics'] == 1]
  critics_to_oscar_rate = critics_winners['won_oscar'].mean()

  # Gather Average Nominations for Winners
  oscar_winners = category_df[category_df['won_oscar'] == 1]
  oscar_winner_average_nomination_count = oscar_winners['total_oscar_nominations'].mean()

  # Gather Average Nominations for Non-Winners
  non_oscar_winners = category_df[category_df['won_oscar'] == 0]
  non_oscar_winner_average_nomination_count = non_oscar_winners['total_oscar_nominations'].mean()

  return {"sag_rate" : sag_to_oscar_rate,
          "globe_rate" : globe_to_oscar_rate,
          "bafta_rate" : bafta_to_oscar_rate,
          "critics_rate" : critics_to_oscar_rate,
          "average_winner_nominations" : oscar_winner_average_nomination_count,
          "average_non_winner_nominations" : non_oscar_winner_average_nomination_count}

In [5]:
nomination_count_weight = 0.04

In [6]:
# Calculates the score of a Best Picture Nominee
def best_picture_score(nominee_row, statistics):

  score = 0

  score += nominee_row['won_pga'] * statistics['pga_rate']
  score += nominee_row['won_dga'] * statistics['dga_rate']
  score += nominee_row['won_bafta'] * statistics['bafta_rate']
  score += nominee_row['won_critics'] * statistics['critics_rate']
  score += nominee_row['total_oscar_nominations'] * nomination_count_weight

  return score

In [7]:
# Calculates the score of a nominee in an Acting Category
def acting_score(nominee_row, statistics):
  score = 0

  score += nominee_row['won_sag'] * statistics['sag_rate']
  score += nominee_row['won_globe'] * statistics['globe_rate']
  score += nominee_row['won_bafta'] * statistics['bafta_rate']
  score += nominee_row['won_critics'] * statistics['critics_rate']
  score += nominee_row['total_oscar_nominations'] * nomination_count_weight

  return score

## Run the Backtest

For every category and every year, hold that year out, compute stats from the remaining years, score that year's nominees, and check whether the model's top pick matches who actually won.

In [8]:
# Each category's DataFrame, which stats/score functions to use, and
# which column identifies a nominee (a film title or a person's name)
categories = [
    ('best_picture', best_picture_df, get_best_picture_statistics, best_picture_score, 'film'),
    ('best_actor', best_actor_df, get_acting_category_statistics, acting_score, 'actor_name'),
    ('best_actress', best_actress_df, get_acting_category_statistics, acting_score, 'actor_name'),
    ('best_supporting_actor', best_supporting_actor_df, get_acting_category_statistics, acting_score, 'actor_name'),
    ('best_supporting_actress', best_supporting_actress_df, get_acting_category_statistics, acting_score, 'actor_name'),
]

all_years = sorted(best_picture_df['year'].unique())

results = []

for test_year in all_years:
    for category_name, category_df, stats_fn, score_fn, name_col in categories:
        # Hold test_year out of the data used to compute stats
        train_df = category_df[category_df['year'] != test_year]
        test_df = category_df[category_df['year'] == test_year]

        stats = stats_fn(train_df)

        # Score every nominee that year, same pattern as predict_picture/predict_acting
        nominee_scores = []
        for i, row in test_df.iterrows():
            score = score_fn(row, stats)
            nominee_scores.append({'name': row[name_col], 'score': score})

        ranked = pandas.DataFrame(nominee_scores).sort_values(by='score', ascending=False)
        predicted = ranked['name'].head(1).to_string(index=False)

        actual_row = test_df[test_df['won_oscar'] == 1]
        actual = actual_row[name_col].head(1).to_string(index=False) if len(actual_row) else None

        results.append({
            'year': test_year,
            'category': category_name,
            'predicted': predicted,
            'actual': actual,
            'correct': predicted == actual,
        })

results_df = pandas.DataFrame(results)
print(f"Backtested {len(results_df)} category-years.")

Backtested 55 category-years.


## How accurate is it?

In [9]:
overall_accuracy = results_df['correct'].mean()
print(f"Overall accuracy: {overall_accuracy:.1%} ({results_df['correct'].sum()} / {len(results_df)} correct)")

print("\nBy category:")
category_accuracy = results_df.groupby('category')['correct'].mean()
for category, accuracy in category_accuracy.items():
    print(f"  {category:<28} {accuracy:.1%}")

Overall accuracy: 78.2% (43 / 55 correct)

By category:
  best_actor                   81.8%
  best_actress                 72.7%
  best_picture                 54.5%
  best_supporting_actor        90.9%
  best_supporting_actress      90.9%


## Full results

In [10]:
results_df.sort_values(['year', 'category']).reset_index(drop=True)

,year,category,predicted,actual,correct
0,2015,best_actor,Eddie Redmayne,Eddie Redmayne,True
1,2015,best_actress,Julianne Moore,Julianne Moore,True
2,2015,best_picture,Birdman or (The Unexpected Virtue of Ignorance),Birdman or (The Unexpected Virtue of Ignorance),True
3,2015,best_supporting_actor,J.K Simmons,J.K Simmons,True
4,2015,best_supporting_actress,Patricia Arquette,Patricia Arquette,True
5,2016,best_actor,Leonardo Dicaprio,Leonardo Dicaprio,True
6,2016,best_actress,Brie Larson,Brie Larson,True
7,2016,best_picture,The Revenant,Spotlight,False
8,2016,best_supporting_actor,Sylvester Stallone,Mark Rylance,False
9,2016,best_supporting_actress,Alicia Vikander,Alicia Vikander,True


## Takeaways

Overall accuracy lands around **78%** across 55 category-years. Best Picture is the hardest category to call (~55%), while the supporting-acting categories are the easiest (~91%) - worth keeping in mind before leaning too hard on any single category's prediction.

One honest caveat: this is only 11 years of data, so this as a reasonable estimate of how the model performs, not a guarantee. 